#Initial Setup

In [1]:
# --------------------------------------------------
# NORTHSTAR URBAN MOBILITY
# Phase 1 - Data Ingestion & Preparation
# --------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', 50)
sns.set_style("darkgrid")

print("Core libraries initialized.")

Core libraries initialized.


#Data Loading

In [2]:
# --------------------------------------------------
# Loading Source CSV Files
# --------------------------------------------------

source_files = {
    'hubs': 'hubs.csv',
    'customers': 'customers.csv',
    'drivers': 'drivers.csv',
    'vehicles': 'vehicles.csv',
    'orders': 'orders.csv',
    'deliveries': 'deliveries.csv',
    'incidents': 'incidents.csv',
    'complaints': 'complaints.csv',
    'app_events': 'app_events.csv'
}

source_data = {}
for label, fname in source_files.items():
    try:
        source_data[label] = pd.read_csv(fname)
        print(f"Imported {label:12} | Rows: {source_data[label].shape[0]:,}")
    except Exception as err:
        print(f"Could not load {fname} → {err}")

Imported hubs         | Rows: 8
Imported customers    | Rows: 650
Imported drivers      | Rows: 170
Imported vehicles     | Rows: 120
Imported orders       | Rows: 1,250
Imported deliveries   | Rows: 950
Imported incidents    | Rows: 280
Imported complaints   | Rows: 320
Imported app_events   | Rows: 640


#Dataset Cleaning

In [3]:
# --------------------------------------------------
# Zone Name Standardization & Basic Cleaning
# --------------------------------------------------

def clean_zone(val):
    if pd.isna(val):
        return val
    cleaned = str(val).strip().title()
    cleaned = cleaned.replace("Riverside", "Riverside").replace("RiverSide", "Riverside")
    cleaned = cleaned.replace("Ctr", "Central")
    return cleaned

for key in ['orders', 'deliveries', 'customers', 'drivers', 'hubs']:
    if key in source_data:
        df = source_data[key]
        for col in df.columns:
            if 'zone' in col.lower():
                df[col] = df[col].apply(clean_zone)
        print(f"Zone standardization completed for {key}")

Zone standardization completed for orders
Zone standardization completed for deliveries
Zone standardization completed for customers
Zone standardization completed for drivers
Zone standardization completed for hubs


#Temporal Data Handling

In [4]:
# --------------------------------------------------
# Date Parsing Across Tables
# --------------------------------------------------

date_fields = {
    'orders': ['order_created_at'],
    'deliveries': ['dispatch_time', 'delivery_completed_at'],
    'complaints': ['created_at'],
    'app_events': ['event_timestamp'],
    'incidents': ['reported_at']
}

for table_key, cols in date_fields.items():
    if table_key in source_data:
        df = source_data[table_key]
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors='coerce')
                print(f"Date parsing done → {table_key}.{c}")

Date parsing done → orders.order_created_at
Date parsing done → deliveries.dispatch_time
Date parsing done → deliveries.delivery_completed_at
Date parsing done → complaints.created_at
Date parsing done → app_events.event_timestamp
Date parsing done → incidents.reported_at


#Export Cleaned Files

In [5]:
# --------------------------------------------------
# Exporting Final Cleaned Version
# --------------------------------------------------

import os
os.makedirs('northstar_ready', exist_ok=True)

for name, df in source_data.items():
    target_file = f'northstar_ready/{name}_ready.csv'
    df.to_csv(target_file, index=False)
    print(f"Exported → {target_file}")

Exported → northstar_ready/hubs_ready.csv
Exported → northstar_ready/customers_ready.csv
Exported → northstar_ready/drivers_ready.csv
Exported → northstar_ready/vehicles_ready.csv
Exported → northstar_ready/orders_ready.csv
Exported → northstar_ready/deliveries_ready.csv
Exported → northstar_ready/incidents_ready.csv
Exported → northstar_ready/complaints_ready.csv
Exported → northstar_ready/app_events_ready.csv


#Package for Download

In [6]:
# --------------------------------------------------
# Create Downloadable Archive
# --------------------------------------------------

from google.colab import files
import shutil

shutil.make_archive('northstar_ready', 'zip', 'northstar_ready')
files.download('northstar_ready.zip')

print("Archive ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archive ready for download.
